# Model Evaluation — Fraud Detection

## Objective

`06_model_experiments.ipynb` compared several models on PR-AUC and picked
a winner by breadth. This notebook goes deep on that one winning model
specifically, in four directions the knowledge layer flags as each being
necessary but not sufficient on their own:

1. **Classification detail** (`19_MODEL_EVALUATION`) — not just a single
   PR-AUC number, but the full precision/recall/ROC picture and what the
   confusion matrix actually implies.
2. **Calibration** (`20_RISK_SCORING`) — a model can rank transactions
   correctly (high AUC) while still being poorly calibrated, meaning its
   probabilities don't mean what they claim to mean. This matters
   enormously once a fixed threshold gets set on top of it.
3. **Business cost** (`12_BUSINESS_PROBLEM`, `25_BUSINESS_EVALUATION`) —
   translating precision/recall into an actual cost-minimizing threshold,
   rather than a statistically convenient one like "maximize F1."
4. **Error analysis** — looking at *which* transactions the model gets
   wrong, not just how many, to generate concrete hypotheses for what
   might improve it later.

**This notebook does not retrain anything.** It evaluates whatever model
`06_model_experiments.ipynb` already chose. If you want to try a
different model, that happens upstream, not here.


## Setup


In [ ]:
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
)
from sklearn.calibration import calibration_curve

import joblib


In [ ]:
train_fe = pd.read_pickle(r'C:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model\data\02_intermediate\train_features.pkl')


In [ ]:
print("train_fe shape:", train_fe.shape)


## Step 1: Reproduce the Feature Set and Split

Same reasoning as `06_model_experiments.ipynb`: this notebook reloads its
own data and rebuilds the identical feature list and time-based split,
so whatever we measure here is measured on the exact same validation
slice the model was selected against — not a slightly different one that
would make the numbers incomparable.


In [ ]:
exclude_cols = {"TransactionID", "isFraud"}
raw_categorical_cols = {"ProductCD", "card4", "card6", "DeviceType",
                         "P_emaildomain", "R_emaildomain"}
raw_identity_cols = {c for c in train_fe.columns if c.startswith("id_")}
exclude_cols |= raw_categorical_cols
exclude_cols |= raw_identity_cols

feature_cols = [c for c in train_fe.columns if c not in exclude_cols]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train_fe[c])]

train_sorted = train_fe.sort_values("TransactionDT").reset_index(drop=True)
split_idx = int(len(train_sorted) * 0.8)

train_split = train_sorted.iloc[:split_idx]
val_split = train_sorted.iloc[split_idx:]

X_train = train_split[feature_cols]
y_train = train_split["isFraud"]
X_val = val_split[feature_cols]
y_val = val_split["isFraud"]

print(f"Val rows: {len(X_val)}  |  Fraud rate in val: {y_val.mean():.4%}")


## Step 2: Load the Model

We try the winning model from `06_model_experiments.ipynb` first. If
that file isn't present, we fall back to the `05_baseline_model.ipynb`
model. If neither exists, we retrain the baseline inline as a last
resort, so this notebook can still run and demonstrate the evaluation
process end to end.


In [ ]:
winning_path = r'C:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model\models\best_experiment_model.joblib'
baseline_path = r'C:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model\models\baseline_logistic_regression.joblib'

artifact = None
source = None

for path, label in [(winning_path, "06_model_experiments winner"), (baseline_path, "05_baseline_model")]:
    try:
        artifact = joblib.load(path)
        source = label
        break
    except FileNotFoundError:
        continue

if artifact is not None:
    model = artifact["model"]
    scaler = artifact.get("scaler")
    print(f"Loaded model from: {source}")
    print(f"Model type: {type(model).__name__}")
else:
    print("No saved model found -- retraining the baseline inline as a fallback.")
    scaler = StandardScaler()
    X_train_scaled_fb = scaler.fit_transform(X_train)
    model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
    model.fit(X_train_scaled_fb, y_train)
    source = "inline fallback retrain"


In [ ]:
X_val_for_model = scaler.transform(X_val) if scaler is not None else X_val
val_probs = model.predict_proba(X_val_for_model)[:, 1]


## Step 3: Classification Report and Confusion Matrix, in Detail

Beyond the summary numbers, we name each confusion-matrix quadrant in
plain terms and compute the rates that matter most for a fraud system
specifically:

- **Recall (a.k.a. True Positive Rate):** of all actual fraud, what
  fraction did we catch?
- **False Positive Rate:** of all legitimate transactions, what fraction
  did we wrongly flag?
- **Precision:** of everything we flagged, what fraction was really
  fraud?


In [ ]:
val_preds = (val_probs >= 0.5).astype(int)

print("Classification report (threshold = 0.5):")
print(classification_report(y_val, val_preds, digits=4))

cm = confusion_matrix(y_val, val_preds)
tn, fp, fn, tp = cm.ravel()

recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan

print(f"\nTrue Positives (fraud caught):      {tp}")
print(f"False Negatives (fraud missed):      {fn}")
print(f"False Positives (legit blocked):     {fp}")
print(f"True Negatives (legit approved):     {tn}")
print(f"\nRecall (fraud caught):   {recall:.4f}")
print(f"False Positive Rate:     {fpr:.4f}")
print(f"Precision:               {precision:.4f}")


In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted: Legit", "Predicted: Fraud"],
            yticklabels=["Actual: Legit", "Actual: Fraud"])
plt.title(f"Confusion Matrix (threshold = 0.5)\nModel: {source}")
plt.tight_layout()
plt.show()


## Step 4: ROC and Precision-Recall Curves

Shown side by side deliberately. As covered in `19_MODEL_EVALUATION`, the
ROC curve can look reassuring on imbalanced data even when the PR curve
reveals the model is far from precise at any usable recall level — seeing
both at once makes that gap visible rather than hidden behind one AUC
number.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fpr_curve, tpr_curve, _ = roc_curve(y_val, val_probs)
roc_auc_val = roc_auc_score(y_val, val_probs)
axes[0].plot(fpr_curve, tpr_curve, color="darkorange", label=f"ROC (AUC = {roc_auc_val:.3f})")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate (Recall)")
axes[0].set_title("ROC Curve")
axes[0].legend()

precisions, recalls, _ = precision_recall_curve(y_val, val_probs)
pr_auc_val = average_precision_score(y_val, val_probs)
axes[1].plot(recalls, precisions, color="firebrick", label=f"PR (AUC = {pr_auc_val:.3f})")
baseline_rate = y_val.mean()
axes[1].axhline(baseline_rate, linestyle="--", color="gray",
                 label=f"No-skill baseline ({baseline_rate:.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()

plt.tight_layout()
plt.show()


**The "no-skill baseline" line on the PR plot** is the fraud rate itself
— the precision a model gets by randomly flagging transactions with no
information at all. Any useful model's PR curve needs to sit well above
that line, not just above zero.


## Step 5: Calibration — Do the Probabilities Mean What They Say?

Per `20_RISK_SCORING`, ranking ability (AUC) and calibration are
independent properties. A model can correctly rank "this transaction is
riskier than that one" while still being wrong about *how* risky, in
absolute terms. We check this by bucketing validation transactions into
probability bins and comparing the model's average predicted probability
in each bin against the bin's actual observed fraud rate.


In [ ]:
prob_true, prob_pred = calibration_curve(y_val, val_probs, n_bins=10, strategy="quantile")

plt.figure(figsize=(6, 6))
plt.plot(prob_pred, prob_true, marker="o", color="steelblue", label=source)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
plt.xlabel("Mean predicted probability (per bin)")
plt.ylabel("Observed fraud rate (per bin)")
plt.title("Calibration Curve")
plt.legend()
plt.tight_layout()
plt.show()


**Reading this plot:** points sitting on the diagonal mean "when the
model says 30% risk, roughly 30% of those transactions really were
fraud" — well calibrated. Points consistently above the diagonal mean the
model is *underconfident* (actual risk is higher than predicted); points
below mean it's *overconfident*. If this model will feed directly into
fixed thresholds in `08_risk_scoring.ipynb`, meaningful miscalibration
here is worth addressing (e.g. with Platt scaling or isotonic regression)
before those thresholds are trusted.


## Step 6: Threshold Sweep

A finer-grained look than the 9-point table in earlier notebooks. We also
identify the threshold that maximizes F1 — a purely statistical balance
point between precision and recall, included here as a *reference*, not
a recommendation. The business-informed threshold comes next, in Step 7,
and the two frequently don't agree.


In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)
sweep_rows = []

for t in thresholds:
    preds_t = (val_probs >= t).astype(int)
    tp_t = ((preds_t == 1) & (y_val == 1)).sum()
    fp_t = ((preds_t == 1) & (y_val == 0)).sum()
    fn_t = ((preds_t == 0) & (y_val == 1)).sum()

    prec_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0.0
    rec_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0.0
    f1_t = (2 * prec_t * rec_t / (prec_t + rec_t)) if (prec_t + rec_t) > 0 else 0.0

    sweep_rows.append({
        "threshold": t, "tp": tp_t, "fp": fp_t, "fn": fn_t,
        "precision": prec_t, "recall": rec_t, "f1": f1_t,
        "flagged": int((preds_t == 1).sum()),
    })

sweep_df = pd.DataFrame(sweep_rows)
best_f1_row = sweep_df.loc[sweep_df["f1"].idxmax()]

print(f"Max-F1 threshold: {best_f1_row['threshold']:.2f}  "
      f"(precision={best_f1_row['precision']:.3f}, recall={best_f1_row['recall']:.3f}, "
      f"f1={best_f1_row['f1']:.3f})")


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(sweep_df["threshold"], sweep_df["precision"], label="Precision", color="steelblue")
plt.plot(sweep_df["threshold"], sweep_df["recall"], label="Recall", color="firebrick")
plt.plot(sweep_df["threshold"], sweep_df["f1"], label="F1", color="darkgreen", linestyle="--")
plt.axvline(best_f1_row["threshold"], color="gray", linestyle=":", label="Max-F1 threshold")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision / Recall / F1 vs. Threshold")
plt.legend()
plt.tight_layout()
plt.show()


## Step 7: The Business-Informed Threshold

Per `12_BUSINESS_PROBLEM`, false negatives and false positives don't cost
the same, and F1 treats them as if they do. Here we assign each an
explicit dollar cost and find the threshold that minimizes *total*
expected cost — not the statistically "balanced" one from Step 6.

**The two numbers below are placeholders.** They are not derived from
this dataset — they need to come from your actual business context (an
average confirmed fraud loss, and an estimated cost of wrongly
blocking/reviewing a legitimate customer, including support time and
lost goodwill). Replace them before trusting the threshold this produces.


In [ ]:
# PLACEHOLDER VALUES -- replace with real estimates before trusting the result
AVERAGE_FRAUD_LOSS = 150.00       # dollars lost, on average, per missed fraud case (a false negative)
COST_PER_FALSE_POSITIVE = 8.00    # dollars, on average, per wrongly blocked legitimate transaction

sweep_df["total_cost"] = (
    sweep_df["fn"] * AVERAGE_FRAUD_LOSS +
    sweep_df["fp"] * COST_PER_FALSE_POSITIVE
)

best_cost_row = sweep_df.loc[sweep_df["total_cost"].idxmin()]

print(f"Cost-minimizing threshold: {best_cost_row['threshold']:.2f}")
print(f"  Precision: {best_cost_row['precision']:.3f}   Recall: {best_cost_row['recall']:.3f}")
print(f"  False negatives: {int(best_cost_row['fn'])}   False positives: {int(best_cost_row['fp'])}")
print(f"  Estimated total cost on this validation set: ${best_cost_row['total_cost']:,.2f}")
print(f"\nFor comparison, max-F1 threshold was {best_f1_row['threshold']:.2f} "
      f"with estimated cost ${sweep_df.loc[sweep_df['threshold'].sub(best_f1_row['threshold']).abs().idxmin(), 'total_cost']:,.2f}")


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(sweep_df["threshold"], sweep_df["total_cost"], color="purple")
plt.axvline(best_cost_row["threshold"], color="green", linestyle=":",
            label=f"Cost-minimizing ({best_cost_row['threshold']:.2f})")
plt.axvline(best_f1_row["threshold"], color="gray", linestyle=":",
            label=f"Max-F1 ({best_f1_row['threshold']:.2f})")
plt.xlabel("Threshold")
plt.ylabel("Estimated total cost ($)")
plt.title("Total Business Cost vs. Threshold")
plt.legend()
plt.tight_layout()
plt.show()


**If the two vertical lines above land in very different places,** that's
not a bug — it's the entire point of `12_BUSINESS_PROBLEM`. F1 doesn't
know that a missed fraud case costs almost 19x more than a false alarm in
the placeholder numbers above; the cost-weighted threshold does.


## Step 8: Review Queue Sizing at the Chosen Threshold

Per `25_BUSINESS_EVALUATION`, a threshold isn't just a statistical choice
— it determines operational load. At the cost-minimizing threshold found
above, how many transactions would actually get flagged?


In [ ]:
chosen_threshold = best_cost_row["threshold"]
n_flagged = int(best_cost_row["flagged"])
pct_flagged = n_flagged / len(y_val)

print(f"At threshold {chosen_threshold:.2f}:")
print(f"  {n_flagged} of {len(y_val)} validation transactions flagged ({pct_flagged:.2%})")
print(
    "\nScale this percentage to real daily/monthly transaction volume to "
    "estimate how large a manual review queue this threshold would "
    "actually generate -- a number worth sanity-checking against your "
    "review team's real capacity before adopting it."
)


## Step 9: Error Analysis — What Is the Model Getting Wrong?

Counts tell us *how much* the model errs; looking at actual rows tells us
*what kind* of mistake it's making, which is far more useful for deciding
what to improve next.

- **Confident false positives:** legitimate transactions the model was
  very sure were fraud. These suggest a feature the model over-relies on.
- **Confident false negatives:** actual fraud the model was very sure was
  legitimate. These suggest a pattern the model hasn't learned at all.


In [ ]:
val_analysis = val_split.copy()
val_analysis["predicted_prob"] = val_probs

false_positives = val_analysis[(val_analysis["isFraud"] == 0) & (val_analysis["predicted_prob"] >= 0.5)]
false_negatives = val_analysis[(val_analysis["isFraud"] == 1) & (val_analysis["predicted_prob"] < 0.5)]

inspect_cols = [c for c in ["TransactionAmt", "card1_amt_zscore", "card1_txn_count_1h",
                             "card1_seconds_since_last_txn", "has_identity",
                             "predicted_prob"] if c in val_analysis.columns]

print(f"{len(false_positives)} confident false positives, {len(false_negatives)} confident false negatives.")
print("\nTop 5 most confident false positives (legit, flagged as high-risk anyway):")
print(false_positives.sort_values("predicted_prob", ascending=False)[inspect_cols].head())

print("\nTop 5 most confident false negatives (fraud, scored as low-risk):")
print(false_negatives.sort_values("predicted_prob")[inspect_cols].head())


**What to do with this table:** look for a shared pattern across the
false negatives specifically — e.g. if most of them have a low
`card1_amt_zscore` despite being fraud, that's a concrete sign the
account-level baseline features aren't capturing a type of fraud that
doesn't look unusual relative to the account's own history (perhaps
account takeover rather than card-testing — see `05_TYPES_OF_FRAUD`).
That's a hypothesis for a *new* feature, to carry back to
`04_feature_engineering.ipynb`, not something this notebook can fix on
its own.


## Step 10: Save the Evaluation Report

A compact, reusable summary — not the full notebook output — saved to
`reports/`, so the key numbers from this evaluation can be referenced
later (in `25_BUSINESS_EVALUATION`, or just to track whether a future
model version actually improved on this one) without re-running the
whole notebook.


In [ ]:
evaluation_summary = {
    "model_source": source,
    "model_type": type(model).__name__,
    "validation_set_size": int(len(y_val)),
    "validation_fraud_rate": float(y_val.mean()),
    "roc_auc": float(roc_auc_val),
    "pr_auc": float(pr_auc_val),
    "max_f1_threshold": float(best_f1_row["threshold"]),
    "max_f1_score": float(best_f1_row["f1"]),
    "cost_assumptions": {
        "average_fraud_loss": AVERAGE_FRAUD_LOSS,
        "cost_per_false_positive": COST_PER_FALSE_POSITIVE,
    },
    "cost_minimizing_threshold": float(best_cost_row["threshold"]),
    "cost_minimizing_precision": float(best_cost_row["precision"]),
    "cost_minimizing_recall": float(best_cost_row["recall"]),
    "cost_minimizing_total_cost_on_val_set": float(best_cost_row["total_cost"]),
    "pct_of_validation_flagged_at_chosen_threshold": float(pct_flagged),
}

report_path = r'C:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model\reports\model_evaluation_summary.json'
with open(report_path, "w") as f:
    json.dump(evaluation_summary, f, indent=2)

print("Saved evaluation summary to:")
print(" ", report_path)
print()
print(json.dumps(evaluation_summary, indent=2))


## Summary

Evaluated the model selected in `06_model_experiments.ipynb` across four
dimensions:

- **Classification detail:** full confusion matrix, recall, false
  positive rate, and precision, beyond the single PR-AUC headline number.
- **Calibration:** checked whether predicted probabilities match observed
  fraud rates — a prerequisite for `08_risk_scoring.ipynb` to be
  meaningful rather than just well-ranked.
- **Business cost:** found a cost-minimizing threshold using explicit
  (placeholder) false-negative and false-positive costs, and showed it
  can genuinely differ from the statistically "balanced" max-F1
  threshold — the central point of `12_BUSINESS_PROBLEM`.
- **Error analysis:** inspected the model's most confident mistakes in
  both directions, to generate concrete hypotheses for future feature
  work rather than leaving "the model is imperfect" unexamined.

**Before trusting the cost-minimizing threshold found here:** replace
`AVERAGE_FRAUD_LOSS` and `COST_PER_FALSE_POSITIVE` with real estimates —
everything downstream of those two numbers is only as trustworthy as they
are.

**Next step:** `08_risk_scoring.ipynb` — turning this model's raw
probabilities into a calibrated, banded risk score a reviewer or
downstream system can act on directly.
